# A&R Analyst Portfolio Project — Part 3
### Combined Scoring Model: Ranking Emerging Artists

**Prepared by:** Manav Desai

Building on Part 2's momentum signals, this section develops a weighted scoring formula to rank emerging artists by scouting potential — combining audience reach with fan engagement/loyalty.

## 1. Loading Data

In [14]:
# Load the emerging artist data generated in Part 2 (avoids re-running the API pipeline)
import pandas as pd

artist_df = pd.read_csv('data/emerging_artists.csv')
display(artist_df)

,artist,listeners,playcount,tier,plays_per_listener
0,Beyoncé,6494338,718545462,Established,110.6
1,Ariana Grande,4617096,1184987084,Established,256.7
2,Tame Impala,4553333,442949472,Established,97.3
3,Olivia Rodrigo,3287911,672410238,Established,204.5
4,Steve Lacy,3237021,303323626,Established,93.7
5,beabadoobee,2747381,345356925,Established,125.7
6,Don Toliver,2645569,300661138,Established,113.6
7,Brent Faiyaz,2120169,247369592,Established,116.7
8,Kehlani,1792038,76144100,Established,42.5
9,Bryson Tiller,1761284,111020928,Established,63.0


In [15]:
# Confirm which columns carried over from Part 2's saved CSV
print(artist_df.columns.tolist())

['artist', 'listeners', 'playcount', 'tier', 'plays_per_listener']


In [16]:
# The 'tier' column wasn't included in Part 2's saved CSV — recreating it here
# using the same logic, so this notebook works independently
def classify_artist(listeners):
    if listeners >= 1_000_000:
        return 'Established'
    elif listeners >= 100_000:
        return 'Emerging'
    else:
        return 'Early-Stage'

artist_df['tier'] = artist_df['listeners'].apply(classify_artist)
print(artist_df.columns.tolist())


['artist', 'listeners', 'playcount', 'tier', 'plays_per_listener']


## 2. Isolating the Actionable Scouting Pool

Filtering to the "Emerging" tier — artists with real audience traction but not yet at superstar saturation. This is the pool most relevant for scouting decisions.

In [17]:
# Filter to Emerging-tier artists — the actionable pool for scouting decisions
emerging_df = artist_df[artist_df['tier'] == 'Emerging']
print(emerging_df.shape)
display(emerging_df)

(13, 5)


,artist,listeners,playcount,tier,plays_per_listener
15,Motionless In White,948096,65243361,Emerging,68.8
16,HUGEL,675559,8744398,Emerging,12.9
17,F3miii,525489,4341253,Emerging,8.3
18,Ella Langley,484532,13002579,Emerging,26.8
19,Bella Kay,438705,11318081,Emerging,25.8
20,Ian Asher,377081,4266170,Emerging,11.3
21,Dexter and The Moonrocks,332222,4807974,Emerging,14.5
22,PIXY,277208,7083835,Emerging,25.6
23,kwn,243771,6336419,Emerging,26.0
24,jigitz,232724,3237473,Emerging,13.9


## 3. Normalizing Signals

Listener count and plays-per-listener are on very different scales. Applying min-max scaling brings both to a comparable 0-1 range before combining them into a single score.

In [18]:
# Normalize listeners to a 0-1 scale using min-max scaling
emerging_df['listeners_norm'] = (emerging_df['listeners'] - emerging_df['listeners'].min()) / (emerging_df['listeners'].max() - emerging_df['listeners'].min())

# Normalize plays_per_listener the same way
emerging_df['engagement_norm'] = (emerging_df['plays_per_listener'] - emerging_df['plays_per_listener'].min()) / (emerging_df['plays_per_listener'].max() - emerging_df['plays_per_listener'].min())

display(emerging_df)

,artist,listeners,playcount,tier,plays_per_listener,listeners_norm,engagement_norm
15,Motionless In White,948096,65243361,Emerging,68.8,1.000000,1.000000
16,HUGEL,675559,8744398,Emerging,12.9,0.660695,0.076033
17,F3miii,525489,4341253,Emerging,8.3,0.473860,0.000000
18,Ella Langley,484532,13002579,Emerging,26.8,0.422869,0.305785
19,Bella Kay,438705,11318081,Emerging,25.8,0.365815,0.289256
20,Ian Asher,377081,4266170,Emerging,11.3,0.289094,0.049587
21,Dexter and The Moonrocks,332222,4807974,Emerging,14.5,0.233245,0.102479
22,PIXY,277208,7083835,Emerging,25.6,0.164753,0.285950
23,kwn,243771,6336419,Emerging,26.0,0.123124,0.292562
24,jigitz,232724,3237473,Emerging,13.9,0.109371,0.092562


## 4. Building the Scouting Score

Combining reach (listener count) and loyalty (plays-per-listener) into a single weighted score, favoring engagement (60%) over raw reach (40%) — reflecting the view that a devoted niche fanbase is a stronger predictor of breakout potential than current audience size alone.

In [20]:
# Combine normalized signals into a single Scouting Score, weighting engagement slightly higher than reach
emerging_df['scouting_score'] = (0.4 * emerging_df['listeners_norm']) + (0.6 * emerging_df['engagement_norm'])

# Sort by score to get our final ranked shortlist
emerging_df = emerging_df.sort_values('scouting_score', ascending=False).reset_index(drop=True)
display(emerging_df)

,artist,listeners,playcount,tier,plays_per_listener,listeners_norm,engagement_norm,scouting_score
0,Motionless In White,948096,65243361,Emerging,68.8,1.000000,1.000000,1.000000
1,Ella Langley,484532,13002579,Emerging,26.8,0.422869,0.305785,0.352619
2,Bella Kay,438705,11318081,Emerging,25.8,0.365815,0.289256,0.319880
3,HUGEL,675559,8744398,Emerging,12.9,0.660695,0.076033,0.309898
4,PIXY,277208,7083835,Emerging,25.6,0.164753,0.285950,0.237471
5,kwn,243771,6336419,Emerging,26.0,0.123124,0.292562,0.224787
6,F3miii,525489,4341253,Emerging,8.3,0.473860,0.000000,0.189544
7,Cameron Whitcomb,176172,4234551,Emerging,24.0,0.038964,0.259504,0.171288
8,Dexter and The Moonrocks,332222,4807974,Emerging,14.5,0.233245,0.102479,0.154785
9,pupsies,144875,3441606,Emerging,23.8,0.000000,0.256198,0.153719


**Takeaway:** With the expanded pool of 13 Emerging-tier artists, Motionless In White still ranks #1 — reinforcing confidence in the score, since the ranking held up after nearly doubling the sample size. HUGEL presents a similar pattern to an earlier finding: despite having the 3rd-highest raw listener count in this pool, its engagement ratio (12.9) is among the lowest, dropping it to 4th overall. This reinforces that the scoring model consistently distinguishes reach from genuine fan loyalty, rather than simply re-ranking by popularity.

## Top Scouting Recommendations

1. **Motionless In White** — highest score on both reach and engagement; strongest overall candidate
2. **Ella Langley** — solid reach with above-average engagement
3. **Bella Kay** — comparable profile to Ella Langley, close third

## Methodology Notes & Limitations

- This scoring model reflects one reasonable weighting choice (60% engagement / 40% reach); different weightings would produce different rankings, and this tradeoff is a judgment call worth discussing openly
- Sample expanded to 13 Emerging-tier artists across 8 genre searches; the top-ranked artist (Motionless In White) held its #1 position after this expansion, supporting the score's stability
- Due to Spotify API restrictions on audio-feature data (documented in Part 2), this model relies solely on Last.fm momentum signals rather than combining them with the audio-characteristic findings from Part 1


## 5. Early-Stage Watchlist

Early-Stage artists (under 100K listeners) are excluded from the main scoring model, since their smaller absolute play/listener counts make ratios like engagement noisier and less statistically stable. Instead, standouts are flagged qualitatively here as names worth monitoring as more data accumulates.

In [21]:
# Flag Early-Stage artists worth watching, sorted by engagement ratio
# (kept separate from the main scoring model due to smaller, noisier sample sizes at this tier)
early_stage_df = artist_df[artist_df['tier'] == 'Early-Stage'].sort_values('plays_per_listener', ascending=False)
display(early_stage_df)

,artist,listeners,playcount,tier,plays_per_listener
29,honestav,90955,2237686,Early-Stage,24.6
28,Avery Anna,95543,1454104,Early-Stage,15.2
30,Maphra,54854,785154,Early-Stage,14.3
31,D A N N Y,36756,280965,Early-Stage,7.6
32,Josh Fawaz,22021,146645,Early-Stage,6.7


**Takeaway:** honestav stands out even at this early stage, with a 24.6 plays-per-listener ratio comparable to established Emerging-tier artists like Ella Langley (26.8) — despite having under 100K listeners. Given the small sample size at this tier, this should be treated as a name worth monitoring rather than a confident recommendation, but it's a promising early signal.